In [ ]:
# Required inputs. Use this checkout's notebooks/config.json and notebook 02 publication.
MODEL_ID = "usecase2"
RUN_ID = ""  # Pipeline execution ID shown in Studio (the final part of its ARN).


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import importlib
import json
import os
import sys

import boto3
import pandas as pd
from botocore.config import Config
from botocore.exceptions import ClientError
from IPython.display import display

REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
sys.path.insert(0, str(REPOSITORY / "mlops-models/src"))


In [ ]:
# Resolve this run's Preprocess job; its inputs also identify any cached producers.
if not MODEL_ID or not RUN_ID or "/" in MODEL_ID or "/" in RUN_ID:
    raise ValueError("Set MODEL_ID and the execution ID (not its full ARN) in the first cell.")
notebook_config = json.loads((REPOSITORY / "notebooks/config.json").read_text())
namespace = notebook_config["namespace"]
publication_path = REPOSITORY / ".runs/source-deploy" / namespace / "publication.json"
if not namespace or not publication_path.is_file():
    raise ValueError("Use the config.json and saved publication from notebook 02 for this deployment.")
publication = json.loads(publication_path.read_text())
execution_arn = publication["pipelines"]["training"]["arn"] + "/execution/" + RUN_ID
session = boto3.Session(region_name=publication["region"])
read_config = Config(connect_timeout=5, read_timeout=15,
                     retries={"mode": "standard", "total_max_attempts": 2})
sm = session.client("sagemaker", config=read_config)
s3 = session.client("s3", config=read_config)
print("Namespace:", namespace, "| Execution:", execution_arn, flush=True)

source_execution = execution_arn
seen_executions = set()
job = None
while source_execution not in seen_executions:
    seen_executions.add(source_execution)
    steps = [step for page in sm.get_paginator("list_pipeline_execution_steps").paginate(
        PipelineExecutionArn=source_execution) for step in page["PipelineExecutionSteps"]
        if step["StepName"] == "Preprocess"]
    if len(steps) != 1:
        raise ValueError(f"No single Preprocess step in {source_execution}.")
    step = steps[0]
    job_arn = step.get("Metadata", {}).get("ProcessingJob", {}).get("Arn")
    if job_arn:
        job = sm.describe_processing_job(ProcessingJobName=job_arn.rsplit("/", 1)[-1])
        break
    source_execution = step.get("CacheHitResult", {}).get("SourcePipelineExecutionArn")
    if not source_execution:
        raise ValueError(f"Preprocess has no job yet (status: {step['StepStatus']}).")
    print("Reading cached Preprocess job from:", source_execution, flush=True)
else:
    raise ValueError("Preprocess cache references form a cycle.")

environment = job.get("Environment", {})
if environment.get("MODEL_ID") != MODEL_ID:
    raise ValueError(f"Job model is {environment.get('MODEL_ID')!r}; check MODEL_ID and RUN_ID.")
inputs = {item["InputName"]: item["S3Input"]["S3Uri"].rstrip("/")
          for item in job["ProcessingInputs"] if "S3Input" in item}
S3_INPUTS = {
    "data": inputs["raw"] + "/data.csv",
    "selection": inputs["selection"] + "/selected_features.csv",
    "parameters": inputs["source"] + f"/mlops_models/models/{MODEL_ID}/parameters.json",
}
training_selection = environment.get("MLOPS_TRAINING_SELECTION", environment.get("MLOPS_TRAINING_MODE", "1"))
training_mode = {"Preliminary models with validation": "1", "Final model with all data": "2"}.get(
    training_selection, training_selection)
if training_mode not in {"1", "2"}:
    raise ValueError(f"Unknown training mode: {training_selection!r}.")
if training_mode == "2":
    metadata_uri = environment.get("MLOPS_MODEL_METADATA_URI", "").strip()
    if not metadata_uri or metadata_uri == "none":
        raise ValueError("This full-training job has no ModelMetadataUri; inspect its execution inputs.")
    S3_INPUTS["metadata"] = metadata_uri
print("Preprocess job:", job["ProcessingJobArn"], "| Status:", job["ProcessingJobStatus"])
print("Training mode:", training_selection)


In [ ]:
# Download inputs unchanged. parameters.json comes from the job's source, not an evaluation output.
work_directory = Path(mkdtemp(prefix="preprocess-training-"))
data_directory = work_directory / "raw"
selection_file = work_directory / "selection/selected_features.csv"
parameter_path = work_directory / "settings" / MODEL_ID / "parameters.json"
output_directory = work_directory / "output"
output_directory.mkdir()
local_files = {
    "data": data_directory / "data.csv",
    "selection": selection_file,
    "parameters": parameter_path,
}
if training_mode == "2":
    local_files["metadata"] = parameter_path.with_name("model_metadata.json")
for name, local_file in local_files.items():
    local_file.parent.mkdir(parents=True, exist_ok=True)
    uri = S3_INPUTS[name]
    address = urlsplit(uri)
    if address.scheme != "s3" or not address.netloc or not address.path.strip("/"):
        raise ValueError(f"Expected an S3 object: {uri}")
    print(f"Downloading {name}: {uri} -> {local_file}", flush=True)
    try:
        s3.download_file(address.netloc, address.path.lstrip("/"), str(local_file))
    except ClientError as error:
        code = error.response["Error"]["Code"]
        raise RuntimeError(f"Cannot download {name}: {uri} (S3 {code}).") from error
    if not local_file.is_file() or local_file.stat().st_size == 0:
        raise ValueError(f"Missing or empty input: {local_file}")

config = json.loads(local_files["metadata"].read_text()) if training_mode == "2" else {}
if training_mode == "2" and (not isinstance(config, dict) or not config):
    raise ValueError("Full training requires a nonempty model metadata object.")
os.environ.update({
    "MODEL_ID": MODEL_ID,
    "MLOPS_TRAINING_MODE": training_mode,
    "MLOPS_TRAINING_SELECTION": training_selection,
    "MLOPS_MODEL_METADATA_URI": S3_INPUTS.get("metadata", "none"),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
})
print("Local output directory:", output_directory)


In [ ]:
from mlops_models.model_base import preprocess

expected_source = REPOSITORY / "mlops-models/src/mlops_models/model_base/preprocess.py"
assert Path(preprocess.__file__).resolve() == expected_source.resolve(), (
    f"Loaded {preprocess.__file__}. Restart the kernel in {REPOSITORY}."
)
preprocess = importlib.reload(preprocess)
print("Preprocessing source:", Path(preprocess.__file__).resolve())

# Rerun this cell after editing the base function; downloads do not repeat.
# config carries saved metadata for full training, or {} for preliminary training.
result_directory = preprocess.process_train(
    data_directory=data_directory,
    selection_file=selection_file,
    output_directory=output_directory,
    config=config,
    parameter_path=parameter_path,
)
print("Prepared CSV:", result_directory / "data.csv")
display(pd.read_csv(result_directory / "data.csv", nrows=10, dtype=str))
result_directory
